<a href="https://colab.research.google.com/github/madhuvasucv-hash/Madhumitha-Vasudevan/blob/main/3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:

import pandas as pd
import math
from collections import Counter

# Calculate Entropy
def entropy(data):
    target = data.iloc[:, -1]
    counts = Counter(target)
    total = len(target)

    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)

    return ent


# Calculate Information Gain
def information_gain(data, attribute):
    total_entropy = entropy(data)
    values = data[attribute].unique()

    weighted_entropy = 0

    for value in values:
        subset = data[data[attribute] == value]
        weighted_entropy += (len(subset) / len(data)) * entropy(subset)

    return total_entropy - weighted_entropy


# ID3 Algorithm
def id3(data, attributes):
    target = data.iloc[:, -1]

    # If all samples belong to one class
    if len(target.unique()) == 1:
        return target.iloc[0]

    # If no attributes remain
    if not attributes:
        return target.mode()[0]

    # Select best attribute
    gains = {
        attr: information_gain(data, attr)
        for attr in attributes
    }

    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]

        remaining = [
            attr for attr in attributes
            if attr != best_attribute
        ]

        tree[best_attribute][value] = id3(
            subset.drop(columns=[best_attribute]),
            remaining
        )

    return tree


# Classification function
def classify(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value in tree[attribute]:
        return classify(tree[attribute][value], sample)

    return "Unknown"


# Training Dataset
data = pd.DataFrame({
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],

    'Play': ['No', 'No', 'Yes', 'Yes',
             'Yes', 'No', 'Yes', 'No',
             'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
})

print("Training Dataset:")
print(data)

# Build Decision Tree
attributes = list(data.columns[:-1])
tree = id3(data, attributes)

print("\nDecision Tree:")
print(tree)

# Classify New Sample
sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = classify(tree, sample)

print("\nNew Sample:", sample)
print("Predicted Result:", result)


Training Dataset:
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Result: No
